# 00 · GSE65391 · RNA_array · download

Banchereau R et al. *Cell* 2016;165:551–565. Whole blood, Illumina HumanHT-12 v4 (GPL10558).
158 children with SLE (924 samples) and healthy children.

Three files are downloaded once into `data/GSE65391/` and `data/ncbi/`:

| file | source |
|---|---|
| `GSE65391_series_matrix.txt.gz` | GEO series matrix: expression values and sample characteristics |
| `GPL10558.soft.gz` | GEO platform annotation: probe to Entrez gene |
| `Homo_sapiens.gene_info.gz` | NCBI gene table: Entrez gene to current symbol |

**Literate Programming.** Each line in the code will have an explanation of what it is doing below.

Prose and code are written together so the notebook can be read as well as run (Knuth DE. Literate
programming. *Comput J* 1984;27:97–111; Rule A et al. Ten simple rules for writing and sharing
computational analyses in Jupyter Notebooks. *PLoS Comput Biol* 2019;15:e1007007). Code used in every notebook is explained here, in notebook 00, and later notebooks refer back.

## Load the project's locations

In [1]:
source("../src/paths.R")

`source()` executes an `R` source file, `../src/paths.R`, used in all the notebooks.

`../src/paths.R` defines where files live.

**Inside `paths.R`**

*Global Variables*
  - `ROOT`: this repository folder. It is the folder that contains `endotypes-transcriptomics.yml`.
  - `DATA`: defined as `data/`
  - `NCBI`: defined as `data/ncbi/`.
  - `SEED` is the random seed, 20260928, used wherever a method draws random numbers, setting the seed for reproducibility.

*Functions*
  - `raw(gse, file)`: Given the study and the file to be downloaded, returns the relative directory path for the provided filename for the download, e.g. raw("GSE65391", "file.gz") returns data/GSE65391/file.gz.

  - `art(gse, file)`: Given the study and the jupyter lab notebook generated results, returns the directory path for the provided filename for storing the result, e.g. art("GSE65391", "file.rds") returns data/run_artifacts/GSE65391/file.rds.

  - `fig(gse, file)`: Given the study and the jupyter lab notebook generated figure, returns the directory path for the provided figure filename for storing the result, e.g. fig("GSE65391", "07_heatmap_hclust.png") returns figures/GSE65391/07_heatmap_hclust.png.

  Each of the three creates a folder if it does not exist.

  - `read_gene_set(name)`: Reads a gene list from the relative `genes/` subdirectory, one symbol per line; lines starting with `#` are notes.

## Load the GEOquery library and allow longer downloads

In [2]:
suppressMessages(library(GEOquery))
options(timeout = 3600)

**Explanation**
- `library(GEOquery)` loads the Bioconductor package that downloads and reads GEO files (Davis S,
  Meltzer PS. GEOquery: a bridge between the Gene Expression Omnibus (GEO) and BioConductor.
  *Bioinformatics* 2007;23:1846–1847).
- `suppressMessages()` hides the messages a package prints when it loads.
- `options(timeout = 3600)` lets a download run for up to 3,600 seconds (one hour). The default is 60
  seconds, too short for a 90 MB file on a slow connection.

## Download the series matrix

In [3]:
series <- raw("GSE65391", "GSE65391_series_matrix.txt.gz")
if (!file.exists(series))
  download.file("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE65nnn/GSE65391/matrix/GSE65391_series_matrix.txt.gz",
                series, mode = "wb")

**Explanation**
- `raw("GSE65391", "GSE65391_series_matrix.txt.gz")` returns the path
  `data/GSE65391/GSE65391_series_matrix.txt.gz`. Nothing is downloaded on this line.
  
- `series <- ...` stores that path in a variable named `series`. The name is our choice.

  
- `if (!file.exists(series))`: download only if the file is not already on disk. `!` means "not". A second run skips the download.
- `download.file(url, destfile, mode = "wb")` is a base R function that copies one file from a web address
  to disk.
  - The address is GEO's copy of the series matrix. GEO stores series in folders of 1,000; `GSE65nnn` is
    the folder for GSE65000 to GSE65999.
  - `series` is where the file is saved.
  - `mode = "wb"`: write (`w`) the bytes exactly as received (`b`, binary). The file is compressed with
    gzip. On Windows, without `b`, R changes line-ending bytes and the file is damaged. On macOS and Linux
    it makes no difference.

## Download the platform annotation

In [4]:
gpl <- raw("GSE65391", "GPL10558.soft.gz")
if (!file.exists(gpl)) invisible(getGEO("GPL10558", destdir = raw("GSE65391")))

**Explanation**
- `gpl` is the path where the platform file will be saved.

- `getGEO("GPL10558", destdir = ...)` is GEOquery's download function. GPL10558 is the Illumina
  HumanHT-12 v4 array; its annotation gives the gene each probe measures. `destdir` is the folder where the file will be saved. GEOquery names the file `GPL10558.soft.gz` itself, which is why `gpl` uses that name.

- `raw("GSE65391")`, with no file name, returns the folder `data/GSE65391/`.

- `getGEO` also returns the table it read. It is not needed here, and `invisible()` stops it being printed.

## Download the NCBI gene table

In [5]:
gene_info <- file.path(NCBI, "Homo_sapiens.gene_info.gz")
dir.create(NCBI, showWarnings = FALSE)
if (!file.exists(gene_info))
  download.file("https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz",
                gene_info, mode = "wb")

**Explanation**
- All three studies use the Homo sapiens gene info table, so it goes in `data/ncbi/`, not in a study folder.

- `file.path(NCBI, "Homo_sapiens.gene_info.gz")` joins the folder and the file name.
- `dir.create(NCBI, showWarnings = FALSE)` creates `data/ncbi/`. If it already exists, nothing happens and no warning is printed.


- The download follows the same pattern as the series matrix.
- The table maps each Entrez gene identifier to its current official symbol. Notebook 02 uses it so that all three studies name genes the same way.

## Check the files

In [6]:
data.frame(file = basename(c(series, gpl, gene_info)),
           MB = round(file.size(c(series, gpl, gene_info)) / 1e6, 1))

file,MB
<chr>,<dbl>
GSE65391_series_matrix.txt.gz,90.8
GPL10558.soft.gz,75.8
Homo_sapiens.gene_info.gz,5.2


**Explanation**
- `c(series, gpl, gene_info)` puts the three paths in one vector.
- `basename()` keeps only the file name from each path.
- `file.size()` gives each file's size in bytes; `/ 1e6` converts to megabytes; `round(..., 1)` keeps one decimal.
- `data.frame(file = ..., MB = ...)` shows the two as a table with named columns.
- Reading the check: a file of a few kilobytes instead of tens of megabytes means a web page was saved in place of the file, and the download should be repeated.